# Orbit Wars Agent


In [ ]:
import kaggle_environments
print("kaggle_environments version:", kaggle_environments.__version__)
print("Offline-safe mode: using runtime-provided packages only.")

## Write submission file

In [ ]:
%%writefile submission.py
import math
from typing import Dict, List, Tuple
from collections import namedtuple

Planet = namedtuple("Planet", ["id", "owner", "x", "y", "radius", "ships", "production"])

SUN_X, SUN_Y = 50.0, 50.0
SUN_RADIUS = 10.0
MAX_SPEED = 6.0
MIN_OFFENSE_SCORE = 0.035
MIN_SEND_SHIPS = 6
MAX_OFFENSE_COMMIT_RATIO_EXPANSION = 0.48
MAX_OFFENSE_COMMIT_RATIO_CONQUERING = 0.42
# Early turns + catch-up: prioritize neutrals and cheap grabs.
EXPANSION_MAX_STEP = 150
# Stay in expansion while small or still behind on ships.
EXPANSION_MAX_PLANETS = 4
EXPANSION_SHIP_RATIO = 0.88  # my_ships < enemy_total * this => still expanding
# Deliberate captures: avoid spamming tiny fleets; prefer one meaningful wave.
MIN_MEANINGFUL_CHIP_RATIO = 0.42  # if partial, must be >= this fraction of remaining_need
EARLY_ORBIT_BONUS_STEPS = 120  # boost grabbing closest dynamic planets early


def _distance(ax: float, ay: float, bx: float, by: float) -> float:
    return math.hypot(ax - bx, ay - by)


def _fleet_speed(ships: int, max_speed: float = MAX_SPEED) -> float:
    ships = max(1, int(ships))
    if ships <= 1:
        return 1.0
    scale = (math.log(ships) / math.log(1000)) ** 1.5
    scale = max(0.0, min(1.0, scale))
    return 1.0 + (max_speed - 1.0) * scale


def _segment_to_point_distance(ax: float, ay: float, bx: float, by: float, px: float, py: float) -> float:
    abx, aby = bx - ax, by - ay
    apx, apy = px - ax, py - ay
    ab_len2 = abx * abx + aby * aby
    if ab_len2 == 0.0:
        return math.hypot(apx, apy)
    t = (apx * abx + apy * aby) / ab_len2
    t = max(0.0, min(1.0, t))
    cx, cy = ax + t * abx, ay + t * aby
    return math.hypot(px - cx, py - cy)


def _line_hits_sun(src: Planet, dst: Planet, safety_margin: float = 0.4) -> bool:
    d = _segment_to_point_distance(src.x, src.y, dst.x, dst.y, SUN_X, SUN_Y)
    return d <= (SUN_RADIUS + safety_margin)


def _trajectory_hits_sun(src: Planet, angle: float, ships: int, max_ticks: int = 90) -> bool:
    speed = _fleet_speed(ships)
    x, y = src.x, src.y
    for _ in range(max_ticks):
        nx = x + math.cos(angle) * speed
        ny = y + math.sin(angle) * speed
        if _segment_to_point_distance(x, y, nx, ny, SUN_X, SUN_Y) <= SUN_RADIUS:
            return True
        x, y = nx, ny
        if x < -2 or x > 102 or y < -2 or y > 102:
            break
    return False


def _launch_intercepts_target(
    src: Planet,
    dst: Planet,
    angle: float,
    ships: int,
    step_now: int,
    angular_velocity: float,
    initial_by_id: Dict[int, Planet],
    max_ticks: int = 120,
) -> bool:
    speed = _fleet_speed(ships)
    x, y = src.x, src.y
    for tick in range(1, max_ticks + 1):
        nx = x + math.cos(angle) * speed
        ny = y + math.sin(angle) * speed

        # Out-of-bounds flights are considered failed launches.
        if nx < 0.0 or nx > 100.0 or ny < 0.0 or ny > 100.0:
            return False

        px, py = _predict_orbit_position(
            dst=dst,
            initial_by_id=initial_by_id,
            step_now=step_now,
            eta=tick,
            angular_velocity=angular_velocity,
        )
        if _segment_to_point_distance(x, y, nx, ny, px, py) <= dst.radius:
            return True

        x, y = nx, ny
    return False


def _safe_launch_angle(
    src: Planet,
    dst: Planet,
    send: int,
    step_now: int,
    angular_velocity: float,
    initial_by_id: Dict[int, Planet],
) -> float | None:
    base_angle = _angle_to_predicted(
        src=src,
        dst=dst,
        send=send,
        step_now=step_now,
        angular_velocity=angular_velocity,
        initial_by_id=initial_by_id,
    )

    # Try base angle first, then small corrections to recover near-misses.
    angle_candidates = [base_angle]
    for delta in (0.015, -0.015, 0.03, -0.03, 0.05, -0.05):
        angle_candidates.append(base_angle + delta)

    for angle in angle_candidates:
        if _trajectory_hits_sun(src, angle, send):
            continue
        if _launch_intercepts_target(
            src=src,
            dst=dst,
            angle=angle,
            ships=send,
            step_now=step_now,
            angular_velocity=angular_velocity,
            initial_by_id=initial_by_id,
        ):
            return angle
    return None


def _eta_turns(src: Planet, dst: Planet, ships: int) -> int:
    travel_dist = max(0.0, _distance(src.x, src.y, dst.x, dst.y) - src.radius - dst.radius)
    speed = _fleet_speed(ships)
    return max(1, math.ceil(travel_dist / max(1e-6, speed)))


def _project_defenders(target: Planet, eta: int, my_player: int) -> int:
    if target.owner == -1:
        return int(target.ships)
    if target.owner == my_player:
        return int(target.ships)
    return int(target.ships + target.production * eta)


def _capture_margin(step_now: int, eta: int, target: Planet, my_player: int) -> int:
    # Early game: small margins to expand quickly.
    # Mid/late game: higher margins to reduce near-miss failures.
    if step_now < 120:
        phase_margin = 1
    elif step_now < 300:
        phase_margin = 2
    else:
        phase_margin = 3

    eta_margin = min(4, max(0, eta // 7))
    prod_margin = 1 if target.owner >= 0 and target.owner != my_player else 0
    high_prod_margin = 1 if target.production >= 4 else 0
    return phase_margin + eta_margin + prod_margin + high_prod_margin


def _local_enemy_exposure(target: Planet, enemy_planets: List[Planet]) -> float:
    exposure = 0.0
    for enemy in enemy_planets:
        d = _distance(target.x, target.y, enemy.x, enemy.y)
        if d < 45.0:
            weight = (45.0 - d) / 45.0
            exposure += (enemy.ships + 2 * enemy.production) * weight
    return exposure


def _is_expansion_phase(
    step_now: int,
    my_planets: List[Planet],
    player: int,
    planets: List[Planet],
) -> bool:
    if step_now < EXPANSION_MAX_STEP:
        return True
    my_ships = sum(int(p.ships) for p in my_planets)
    enemy_ships = sum(int(p.ships) for p in planets if p.owner >= 0 and p.owner != player)
    if len(my_planets) <= EXPANSION_MAX_PLANETS:
        return True
    if enemy_ships > 0 and my_ships < enemy_ships * EXPANSION_SHIP_RATIO:
        return True
    return False


def _planet_is_orbiting(dst: Planet, initial_by_id: Dict[int, Planet]) -> bool:
    ip = initial_by_id.get(dst.id)
    return ip is not None and _is_orbiting(ip)


def _target_score_v2(
    src: Planet,
    dst: Planet,
    player: int,
    trial_send: int,
    eta: int,
    defenders: int,
    enemy_planets: List[Planet],
    expansion: bool,
    initial_by_id: Dict[int, Planet],
    step_now: int,
) -> float:
    d = _distance(src.x, src.y, dst.x, dst.y)
    if d <= 0.0:
        return -1e9

    orbit_bonus = 0.0
    value_gain = 8.0 * float(dst.production)
    owner_bonus = 6.0 if dst.owner >= 0 and dst.owner != player else 0.0
    neutral_bonus = 2.0 if dst.owner == -1 else 0.0

    capture_cost = 0.30 * defenders
    distance_cost = 0.11 * d
    eta_cost = 0.65 * eta
    exposure_cost = 0.06 * _local_enemy_exposure(dst, enemy_planets)

    # Penalize huge over-send by this source.
    oversend_cost = 0.20 * max(0.0, float(trial_send - defenders))

    if expansion:
        # Expansion: neutrals and fast grabs; de-prioritize costly enemy strikes.
        if dst.owner == -1:
            value_gain *= 1.4
            neutral_bonus *= 1.5
            distance_cost *= 0.82
        elif dst.owner >= 0 and dst.owner != player:
            owner_bonus *= 0.35
            capture_cost *= 1.25
            exposure_cost *= 1.35
            eta_cost *= 1.12

    # Orbiting neutrals: prioritize early (closest dynamic often wins openings).
    if dst.owner == -1 and _planet_is_orbiting(dst, initial_by_id):
        orbit_bonus = 5.5 if step_now < EARLY_ORBIT_BONUS_STEPS else 2.5
        distance_cost *= 0.88

    return value_gain + owner_bonus + neutral_bonus + orbit_bonus - capture_cost - distance_cost - eta_cost - exposure_cost - oversend_cost


def _angle_to(src: Planet, dst: Planet) -> float:
    return math.atan2(dst.y - src.y, dst.x - src.x)


def _is_orbiting(initial_p: Planet) -> bool:
    orbital_radius = _distance(initial_p.x, initial_p.y, SUN_X, SUN_Y)
    return (orbital_radius + initial_p.radius) < 50.0


def _predict_orbit_position(
    dst: Planet,
    initial_by_id: Dict[int, Planet],
    step_now: int,
    eta: int,
    angular_velocity: float,
) -> Tuple[float, float]:
    initial_p = initial_by_id.get(dst.id)
    if initial_p is None or not _is_orbiting(initial_p):
        return dst.x, dst.y

    # Use initial position + global angular velocity to estimate future position.
    r = _distance(initial_p.x, initial_p.y, SUN_X, SUN_Y)
    start_angle = math.atan2(initial_p.y - SUN_Y, initial_p.x - SUN_X)
    future_angle = start_angle + angular_velocity * (step_now + eta)
    return SUN_X + r * math.cos(future_angle), SUN_Y + r * math.sin(future_angle)


def _angle_to_predicted(
    src: Planet,
    dst: Planet,
    send: int,
    step_now: int,
    angular_velocity: float,
    initial_by_id: Dict[int, Planet],
) -> float:
    # Two-pass refinement: estimate ETA, predict position, recompute ETA once.
    eta1 = _eta_turns(src, dst, send)
    px1, py1 = _predict_orbit_position(dst, initial_by_id, step_now, eta1, angular_velocity)
    pseudo1 = Planet(dst.id, dst.owner, px1, py1, dst.radius, dst.ships, dst.production)
    eta2 = _eta_turns(src, pseudo1, send)
    px2, py2 = _predict_orbit_position(dst, initial_by_id, step_now, eta2, angular_velocity)
    return math.atan2(py2 - src.y, px2 - src.x)


def _dynamic_reserve(src: Planet, my_planets: List[Planet], enemy_planets: List[Planet]) -> int:
    # Base reserve scales with production so stronger planets keep more defenders.
    base_reserve = max(6, 4 + 2 * int(src.production))
    if not enemy_planets:
        return base_reserve

    min_enemy_dist = float("inf")
    local_pressure = 0.0
    for enemy in enemy_planets:
        d = _distance(src.x, src.y, enemy.x, enemy.y)
        if d < min_enemy_dist:
            min_enemy_dist = d
        if d < 35.0:
            # Nearby enemy ships and production both increase required garrison.
            weight = (35.0 - d) / 35.0
            local_pressure += (enemy.ships + 2 * enemy.production) * weight

    proximity_bonus = max(0, int((22.0 - min_enemy_dist) / 3.0))
    pressure_bonus = int(local_pressure / 8.0)
    fragile_empire_bonus = 3 if len(my_planets) <= 2 else 0
    return base_reserve + proximity_bonus + pressure_bonus + fragile_empire_bonus


def _defense_first_reinforcements(
    my_planets: List[Planet],
    available_by_source: Dict[int, int],
    reserve_by_source: Dict[int, int],
    step_now: int,
    angular_velocity: float,
    initial_by_id: Dict[int, Planet],
) -> List[List[float]]:
    moves: List[List[float]] = []

    threatened: List[Tuple[int, Planet]] = []
    for p in my_planets:
        deficit = max(0, reserve_by_source[p.id] - int(p.ships))
        if deficit > 0:
            threatened.append((deficit, p))
    threatened.sort(key=lambda x: x[0], reverse=True)

    for deficit, target in threatened:
        if deficit <= 0:
            continue

        donors = []
        for donor in my_planets:
            if donor.id == target.id:
                continue
            avail = available_by_source.get(donor.id, 0)
            if avail <= 0:
                continue
            d = _distance(donor.x, donor.y, target.x, target.y)
            if d > 42.0:
                continue
            donors.append((d, donor))
        donors.sort(key=lambda x: x[0])

        for _, donor in donors:
            avail = available_by_source.get(donor.id, 0)
            if avail <= 0 or deficit <= 0:
                continue
            send = min(avail, deficit)
            if send < 4:
                continue
            if _line_hits_sun(donor, target):
                continue

            angle = _safe_launch_angle(
                src=donor,
                dst=target,
                send=send,
                step_now=step_now,
                angular_velocity=angular_velocity,
                initial_by_id=initial_by_id,
            )
            if angle is None:
                continue
            moves.append([int(donor.id), float(angle), int(send)])
            available_by_source[donor.id] = avail - send
            deficit -= send

    return moves


def agent(obs):
    player = obs.get("player", 0) if isinstance(obs, dict) else obs.player
    raw_planets = obs.get("planets", []) if isinstance(obs, dict) else obs.planets
    raw_initial_planets = obs.get("initial_planets", []) if isinstance(obs, dict) else obs.initial_planets
    step_now = obs.get("step", 0) if isinstance(obs, dict) else obs.step
    angular_velocity = obs.get("angular_velocity", 0.0) if isinstance(obs, dict) else obs.angular_velocity
    planets = [Planet(*p) for p in raw_planets]
    initial_by_id: Dict[int, Planet] = {p[0]: Planet(*p) for p in raw_initial_planets}

    my_planets = [p for p in planets if p.owner == player]
    targets = [p for p in planets if p.owner != player]
    enemy_planets = [p for p in planets if p.owner >= 0 and p.owner != player]
    if not my_planets or not targets:
        return []

    expansion = _is_expansion_phase(step_now, my_planets, player, planets)
    commit_ratio = MAX_OFFENSE_COMMIT_RATIO_EXPANSION if expansion else MAX_OFFENSE_COMMIT_RATIO_CONQUERING

    reserve_by_source: Dict[int, int] = {
        p.id: _dynamic_reserve(p, my_planets, enemy_planets) for p in my_planets
    }
    available_by_source: Dict[int, int] = {
        p.id: max(0, int(p.ships) - reserve_by_source[p.id]) for p in my_planets
    }

    target_by_id: Dict[int, Planet] = {t.id: t for t in targets}
    target_allocated: Dict[int, int] = {t.id: 0 for t in targets}
    moves: List[List[float]] = _defense_first_reinforcements(
        my_planets=my_planets,
        available_by_source=available_by_source,
        reserve_by_source=reserve_by_source,
        step_now=step_now,
        angular_velocity=angular_velocity,
        initial_by_id=initial_by_id,
    )

    # Directional offense: one launch per source planet, limited target spread.
    offensive_sources = [p for p in my_planets if available_by_source.get(p.id, 0) >= MIN_SEND_SHIPS]
    offensive_sources.sort(key=lambda p: available_by_source[p.id], reverse=True)
    # Expansion: slightly wider target spread for neutrals; conquering: tighter focus.
    if expansion:
        max_distinct_targets = max(3, min(6, len(offensive_sources)))
    else:
        max_distinct_targets = max(2, len(offensive_sources) // 2)
    attacked_target_ids = set()
    total_offense_budget = int(sum(available_by_source.get(p.id, 0) for p in offensive_sources) * commit_ratio)
    offense_sent_so_far = 0

    for src in offensive_sources:
        if offense_sent_so_far >= total_offense_budget:
            break
        available = available_by_source[src.id]
        if available < MIN_SEND_SHIPS:
            continue

        best_choice = None
        best_score = -1.0
        for dst in targets:
            if _line_hits_sun(src, dst):
                continue

            remaining_budget = max(0, total_offense_budget - offense_sent_so_far)
            # Fix send size to capture need (iterative ETA/speed), not a blind min-8 trial.
            send_est = max(1, min(available, remaining_budget))
            eta = 1
            defenders = 0
            need_total = 0
            remaining_need = 0
            for _ in range(6):
                eta = _eta_turns(src, dst, send_est)
                defenders = _project_defenders(dst, eta, player)
                margin = _capture_margin(step_now=step_now, eta=eta, target=dst, my_player=player)
                need_total = defenders + margin
                remaining_need = need_total - target_allocated[dst.id]
                if remaining_need <= 0:
                    break
                new_send = min(available, remaining_need, remaining_budget)
                if new_send == send_est:
                    send_est = new_send
                    break
                send_est = new_send

            if remaining_need <= 0:
                continue

            send = min(available, remaining_need, remaining_budget)
            if send < MIN_SEND_SHIPS:
                continue
            # Deliberate waves only: full capture, or a large enough chunk (no "2s until dead").
            if send < remaining_need:
                if remaining_need > 10 and send < remaining_need * MIN_MEANINGFUL_CHIP_RATIO:
                    continue

            # Prevent spraying across too many targets in one turn.
            if dst.id not in attacked_target_ids and len(attacked_target_ids) >= max_distinct_targets:
                continue

            trial_send = send
            base_score = _target_score_v2(
                src=src,
                dst=dst,
                player=player,
                trial_send=trial_send,
                eta=eta,
                defenders=defenders,
                enemy_planets=enemy_planets,
                expansion=expansion,
                initial_by_id=initial_by_id,
                step_now=step_now,
            )
            saturation_penalty = 1.0 + (target_allocated[dst.id] / max(1.0, need_total))
            directional_score = base_score / saturation_penalty
            if directional_score < MIN_OFFENSE_SCORE:
                continue

            if directional_score > best_score:
                best_score = directional_score
                best_choice = (dst, send)

        if best_choice is None:
            continue

        dst, send = best_choice
        angle = _safe_launch_angle(
            src=src,
            dst=dst,
            send=send,
            step_now=step_now,
            angular_velocity=angular_velocity,
            initial_by_id=initial_by_id,
        )
        if angle is None:
            continue
        moves.append([int(src.id), float(angle), int(send)])
        available_by_source[src.id] -= send
        target_allocated[dst.id] += send
        attacked_target_ids.add(dst.id)
        offense_sent_so_far += send

    return moves

## Optional testing (interactive only)

Leave this commented for submission commits.

In [ ]:
from kaggle_environments import make
import kaggle_environments

print("kaggle_environments version:", kaggle_environments.__version__)

try:
    env = make("orbit_wars", debug=True)
except Exception as e:
    print("orbit_wars unavailable in this runtime:", e)
    print("Submission path is still valid: submission.py generation works.")
    print("Use a runtime/session where orbit_wars is enabled for visual replay.")
else:
    env.run(["submission.py", "random"])

    final = env.steps[-1]
    for i, s in enumerate(final):
        print(f"Player {i}: reward={s.reward}, status={s.status}")

    env.render(mode="ipython", width=900, height=700)

    with open("replay.html", "w", encoding="utf-8") as f:
        f.write(env.render(mode="html"))
    print("Saved replay.html")